In [1]:
# Import the Google Colab drive module to interact with your cloud storage
from google.colab import drive
# Mount your personal Google Drive directory at the '/content/gdrive' path
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
# Import foundational data processing libraries
import pandas as pd
import numpy as np
# Import PyTorch deep learning framework
import torch
# Import standard splitting tool from scikit-learn
from sklearn.model_selection import train_test_split
# Import Hugging Face components for model training, dataset handling, and tokenization
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

In [3]:
# Load the cleaned clinical student mental health dataset from Google Drive
student_df = pd.read_csv('/content/gdrive/MyDrive/mental_health_prediction/cleaned_student_dataset.csv')
# Load the cleaned online community Reddit dataset from Google Drive
reddit_df = pd.read_csv('/content/gdrive/MyDrive/mental_health_prediction/cleaned_reddit_dataset.csv')

In [4]:
# View the first 5 records of the student dataset to check its columns and structure
student_df.head()

,Gender,Age,Academic Pressure,Work Pressure,CGPA,Sleep Duration,Dietary Habits,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression,serialized_text
0,Male,33,5.0,0.0,4.49,'5-6 hours',Healthy,3.0,1.0,0,1,"A 33-year-old Male student with CGPA 4.49, 5.0..."
1,Female,24,2.0,0.0,2.95,'5-6 hours',Moderate,3.0,2.0,1,0,"A 24-year-old Female student with CGPA 2.95, 2..."
2,Male,31,3.0,0.0,3.52,'Less than 5 hours',Healthy,9.0,1.0,1,0,"A 31-year-old Male student with CGPA 3.52, 3.0..."
3,Female,28,3.0,0.0,2.80,'7-8 hours',Moderate,4.0,5.0,1,1,"A 28-year-old Female student with CGPA 2.8, 3...."
4,Female,25,4.0,0.0,4.07,'5-6 hours',Moderate,1.0,1.0,0,0,"A 25-year-old Female student with CGPA 4.07, 4..."


In [5]:
# Drop the pre-existing serialized text column so we can construct a fresh representation
student_df = student_df.drop(['serialized_text'], axis=1)

In [6]:
def serialize_student_row(row):
    """
    Converts your specific student columns into a structured, clinical-style text sentence.
    This bridges the gap between tabular features and textual pre-training models.
    """
    # Map rating integers to descriptive descriptive text levels
    pressure_map = {0: 'no', 1: 'very low', 2: 'low', 3: 'moderate', 4: 'high', 5: 'extreme'}

    # Convert float inputs safely to integer strings
    try:
        academic_pressure = pressure_map.get(int(float(row.get('Academic Pressure', 3))), 'moderate')
        work_pressure = pressure_map.get(int(float(row.get('Work Pressure', 0))), 'low')
        financial_stress  = pressure_map.get(int(float(row.get('Financial Stress', 0))), 'low')
        family_history = 'Yes' if int(float(row.get('Family History of Mental Illness', 0))) == 1 else 'No'
    except:
        # Fallback values if data conversion or parsing encounters an issue
        academic_pressure, work_pressure, financial_stress, family_history = 'moderate', 'low', 'low', 'No'

    # Piece together the final clinical descriptive sequence
    text = (
        f"[CLINICAL] Subject is a {int(float(row.get('Age', 20)))}-year-old {row.get('Gender', 'Student')}. "
        f"Academic standing: CGPA of {row.get('CGPA', 0.0)}. "
        f"They report {academic_pressure} academic pressure and {work_pressure} work pressure. "
        f"Sleep duration: {row.get('Sleep Duration', 'unknown')}. "
        f"Dietary habits: {row.get('Dietary Habits', 'unknown')}. "
        f"Financial stress is {financial_stress}. "
        f"Family history of mental illness: {family_history}."
    )
    return text

In [7]:
# Apply the clinical narrative translation function row-by-row onto our student dataframe
student_df['text'] = student_df.apply(serialize_student_row, axis=1)

In [8]:
# Inspect categorical distributions of target labels for both sources
print("Student Depression labels:")
print(student_df['Depression'].value_counts(dropna=False))

print("\nReddit binary labels:")
print(reddit_df['binary_label'].value_counts(dropna=False))

Student Depression labels:
Depression
1    16336
0    11565
Name: count, dtype: int64

Reddit binary labels:
binary_label
1    26056
0    16343
Name: count, dtype: int64


In [9]:
# Print target labels to confirm counts match and sanity-check data loading
print("Student Depression labels:")
print(student_df['Depression'].value_counts(dropna=False))

print("\nReddit binary labels:")
print(reddit_df['binary_label'].value_counts(dropna=False))

Student Depression labels:
Depression
1    16336
0    11565
Name: count, dtype: int64

Reddit binary labels:
binary_label
1    26056
0    16343
Name: count, dtype: int64


In [10]:
# Display a sample of 10 student records to verify successful string serialization
print(student_df[['Depression', 'text']].sample(10, random_state=42).to_string(index=False))

 Depression                                                                                                                                                                                                                                                                        text
          0       [CLINICAL] Subject is a 29-year-old Female. Academic standing: CGPA of 4.26. They report low academic pressure and no work pressure. Sleep duration: 'More than 8 hours'. Dietary habits: Healthy. Financial stress is extreme. Family history of mental illness: No.
          0             [CLINICAL] Subject is a 28-year-old Male. Academic standing: CGPA of 2.78. They report low academic pressure and no work pressure. Sleep duration: '7-8 hours'. Dietary habits: Unhealthy. Financial stress is very low. Family history of mental illness: Yes.
          0 [CLINICAL] Subject is a 34-year-old Male. Academic standing: CGPA of 2.56. They report moderate academic pressure and no work pressure. Sleep durati

In [11]:
# Display a sample of 10 cleaned Reddit posts to review online discussion content
print(reddit_df[['binary_label', 'clean_text']].sample(10, random_state=42).to_string(index=False))

 binary_label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              clean_text
            1 tyson, fury and countless other champions had to get through it why would it be any easier for me, this d actually hurted me so much

In [12]:
# Inspect the label frequency distributions once more prior to dataset merging
print("Student Depression labels:")
print(student_df['Depression'].value_counts(dropna=False))

print("\nReddit binary labels:")
print(reddit_df['binary_label'].value_counts(dropna=False))

Student Depression labels:
Depression
1    16336
0    11565
Name: count, dtype: int64

Reddit binary labels:
binary_label
1    26056
0    16343
Name: count, dtype: int64


In [13]:
# Remove records with missing values in either text or labels inside Reddit subset
reddit_df = reddit_df.dropna(
    subset=['clean_text', 'binary_label']
).copy()

# Print the shapes to ensure cleanup was successful and review sample counts
print("Student dataset:", student_df.shape)
print("Reddit dataset:", reddit_df.shape)

print("\nRemaining missing values:")
print(reddit_df[['clean_text', 'binary_label']].isnull().sum())

Student dataset: (27901, 12)
Reddit dataset: (42398, 2)

Remaining missing values:
clean_text      0
binary_label    0
dtype: int64


In [14]:
# Re-verify cleanup steps and ensure no missing values persist
reddit_df = reddit_df.dropna(
    subset=['clean_text', 'binary_label']
).copy()

print("Student dataset:", student_df.shape)
print("Reddit dataset:", reddit_df.shape)

print("\nRemaining missing values:")
print(reddit_df[['clean_text', 'binary_label']].isnull().sum())

Student dataset: (27901, 12)
Reddit dataset: (42398, 2)

Remaining missing values:
clean_text      0
binary_label    0
dtype: int64


In [15]:
# Align student features, select necessary columns, and rename targets to 'labels'
student_subset = student_df[['text', 'Depression']].rename(
    columns={'Depression': 'labels'}
).copy()

# Align Reddit features, select necessary columns, and rename targets to 'labels'
reddit_subset = reddit_df[['clean_text', 'binary_label']].rename(
    columns={
        'clean_text': 'text',
        'binary_label': 'labels'
    }
).copy()

# Add metadata identifying origin source before fusion
student_subset['source'] = 'student'
reddit_subset['source'] = 'reddit'

# Concatenate both subsets together into a unified dataframe
full_dataset = pd.concat(
    [student_subset, reddit_subset],
    axis=0,
    ignore_index=True
)

# Perform a random shuffle across all samples to ensure unbiased splits
full_dataset = full_dataset.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Output metadata summaries of final unified dataset
print("Fused dataset shape:", full_dataset.shape)
print("\nLabel distribution:")
print(full_dataset['labels'].value_counts())

print("\nDataset source distribution:")
print(full_dataset['source'].value_counts())

Fused dataset shape: (70299, 3)

Label distribution:
labels
1    42391
0    27908
Name: count, dtype: int64

Dataset source distribution:
source
reddit     42398
student    27901
Name: count, dtype: int64


In [16]:
from sklearn.model_selection import train_test_split

# Stratify based on a composite key of origin-source and ground-truth label
stratify_col = (
    full_dataset['source'].astype(str)
    + '_'
    + full_dataset['labels'].astype(str)
)

# Hold out 70% of entire set for training, remaining 30% for validation/testing
train_df, temp_df = train_test_split(
    full_dataset,
    test_size=0.30,
    random_state=42,
    stratify=stratify_col
)

# Create stratification key for the temporary split subset
temp_stratify = (
    temp_df['source'].astype(str)
    + '_'
    + temp_df['labels'].astype(str)
)

# Split remaining 30% evenly into 15% validation and 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_stratify
)

# Clean index formatting on splits
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Display split dimensions and validation/test balance checks
print("Training set:", train_df.shape)
print("Validation set:", val_df.shape)
print("Testing set:", test_df.shape)

print("\nTraining distribution:")
print(pd.crosstab(train_df['source'], train_df['labels']))

print("\nValidation distribution:")
print(pd.crosstab(val_df['source'], val_df['labels']))

print("\nTesting distribution:")
print(pd.crosstab(test_df['source'], test_df['labels']))

Training set: (49209, 3)
Validation set: (10545, 3)
Testing set: (10545, 3)

Training distribution:
labels       0      1
source               
reddit   11440  18238
student   8096  11435

Validation distribution:
labels      0     1
source             
reddit   2452  3909
student  1734  2450

Testing distribution:
labels      0     1
source             
reddit   2451  3908
student  1735  2451


In [17]:
# Print random samples of Reddit posts that do not represent mental-health concerns (label 0)
print("Reddit examples with label 0:")
display(
    reddit_df[reddit_df['binary_label'] == 0][['binary_label', 'clean_text']].sample(5, random_state=42)
)

# Print random samples of Reddit posts that represent active mental-health concerns (label 1)
print("\nReddit examples with label 1:")
display(
    reddit_df[reddit_df['binary_label'] == 1][['binary_label', 'clean_text']].sample(5, random_state=42)
)

Reddit examples with label 0:


,binary_label,clean_text
40581,0,spring is finally here cherry blossom galore s...
39577,0,is so jeles argh sy pn mau jln sm kau jg saddd...
476,0,cuet life fucked up
37978,0,stephenkruiser that s horrible sorryto hear that
30835,0,it sure is.



Reddit examples with label 1:


,binary_label,clean_text
24721,1,"i have been working since 17, i am 18 now and ..."
8214,1,"the affect, suicide, self-injury, and social t..."
14399,1,i have had a bad day at work. its not unusual ...
12760,1,this is just a long rant. i am really at my wi...
9002,1,it is difficult for me to confront a simple pr...


In [18]:
from huggingface_hub import login
login()

In [19]:
from transformers import AutoTokenizer
from datasets import Dataset

# Define Model identifier targeting pretrained clinical RoBERTa base weights
model_id = "mental/mental-roberta-base"

# Load corresponding pretrained Hugging Face tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Tokenizer loaded successfully!")

config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Tokenizer loaded successfully!


In [20]:
# Convert training pandas DataFrame into Hugging Face Dataset format
train_dataset = Dataset.from_pandas(
    train_df[['text', 'labels']].reset_index(drop=True)
)

# Convert validation pandas DataFrame into Hugging Face Dataset format
val_dataset = Dataset.from_pandas(
    val_df[['text', 'labels']].reset_index(drop=True)
)

# Convert test pandas DataFrame into Hugging Face Dataset format
test_dataset = Dataset.from_pandas(
    test_df[['text', 'labels']].reset_index(drop=True)
)

# Print datasets metadata info
print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'labels'],
    num_rows: 49209
})
Dataset({
    features: ['text', 'labels'],
    num_rows: 10545
})
Dataset({
    features: ['text', 'labels'],
    num_rows: 10545
})


In [21]:
def tokenize_function(examples):
    # Helper to clean, segment, truncate, and pad texts up to 512 tokens
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

# Map tokenization function across training subset
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

# Map tokenization function across validation subset
tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True
)

# Map tokenization function across testing subset
tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/49209 [00:00<?, ? examples/s]

Map:   0%|          | 0/10545 [00:00<?, ? examples/s]

Map:   0%|          | 0/10545 [00:00<?, ? examples/s]

In [22]:
# Inspect the features of tokenized datasets to verify token mappings
print("Training:", tokenized_train)
print("Validation:", tokenized_val)
print("Testing:", tokenized_test)

# Inspect a single entry structure to make sure input_ids match model requirements
print("\nTokenized example:")
print(tokenized_train[0])

Training: Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 49209
})
Validation: Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 10545
})
Testing: Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 10545
})

Tokenized example:
{'text': 'finished making chocolate and foot really hurt', 'labels': 0, 'input_ids': [0, 40544, 442, 7548, 8, 2767, 269, 2581, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [23]:
# Remove un-tokenized raw text columns so that trainer gets clean tensors
tokenized_train = tokenized_train.remove_columns(['text'])
tokenized_val = tokenized_val.remove_columns(['text'])
tokenized_test = tokenized_test.remove_columns(['text'])

print(tokenized_train)
print(tokenized_val)
print(tokenized_test)

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 49209
})
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 10545
})
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 10545
})


In [24]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):
    # De-structure predicted logits and truth labels
    logits, labels = eval_pred

    # Retrieve predictions via argmax over prediction dimensions
    predictions = np.argmax(logits, axis=-1)

    # Compute classic binary metrics focusing on positive case class 1
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='binary',
        pos_label=1,
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [25]:
# Customized Trainer that adds class weights to combat labeling imbalances
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Penalize misclassifying class 1 twice as heavily as class 0
        weights = torch.tensor([1.0, 2.0], device=logits.device)

        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

# Initialize classification model architecture loaded with pre-trained weights
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2
)

# Define the directory pathways and configuration args for fine-tuning

training_args = TrainingArguments(
    output_dir="/content/gdrive/MyDrive/mental_health_prediction/unified_results",
    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    weight_decay=0.01,

    metric_for_best_model="f1",
    greater_is_better=True,
    load_best_model_at_end=True,

    report_to="none"
)


# Build the final trainer class with our customized weighted loss functionality
trainer = WeightedLossTrainer(
    model=model,
    compute_metrics=compute_metrics,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val
)

# Begin fine-tuning iteration
trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: mental/mental-roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9228, training_loss=0.402261128816245, metrics={'train_runtime': 3964.6808, 'train_samples_per_second': 37.236, 'train_steps_per_second': 2.328, 'total_flos': 3.884229576963072e+16, 'train_loss': 0.402261128816245, 'epoch': 3.0})

In [26]:
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

# Confirm which checkpoint was selected
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation F1:", trainer.state.best_metric)

# Generate predictions on the untouched test set
test_output = trainer.predict(tokenized_test)

predictions = np.argmax(test_output.predictions, axis=-1)

# Keep the original test data and attach predictions
results_df = test_df.reset_index(drop=True).copy()
results_df['predicted_label'] = predictions

# Overall test performance
print("\n========== OVERALL TEST RESULTS ==========")

print(classification_report(
    results_df['labels'],
    results_df['predicted_label'],
    labels=[0, 1],
    target_names=[
        'No identified concern',
        'Mental-health concern'
    ],
    digits=4,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    results_df['labels'],
    results_df['predicted_label'],
    labels=[0, 1]
))

# Evaluate student and Reddit separately
for source in ['student', 'reddit']:

    source_df = results_df[
        results_df['source'] == source
    ]

    print(f"\n========== {source.upper()} TEST RESULTS ==========")

    print(classification_report(
        source_df['labels'],
        source_df['predicted_label'],
        labels=[0, 1],
        target_names=[
            'No identified concern',
            'Mental-health concern'
        ],
        digits=4,
        zero_division=0
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(
        source_df['labels'],
        source_df['predicted_label'],
        labels=[0, 1]
    ))

Best checkpoint: /content/gdrive/MyDrive/mental_health_prediction/unified_results/checkpoint-9228
Best validation F1: 0.9246330519431135



========== OVERALL TEST RESULTS ==========
                       precision    recall  f1-score   support

No identified concern     0.9144    0.8323    0.8714      4186
Mental-health concern     0.8958    0.9487    0.9215      6359

             accuracy                         0.9025     10545
            macro avg     0.9051    0.8905    0.8965     10545
         weighted avg     0.9032    0.9025    0.9016     10545

Confusion Matrix:
[[3484  702]
 [ 326 6033]]

========== STUDENT TEST RESULTS ==========
                       precision    recall  f1-score   support

No identified concern     0.7988    0.6225    0.6997      1735
Mental-health concern     0.7689    0.8890    0.8246      2451

             accuracy                         0.7785      4186
            macro avg     0.7838    0.7558    0.7622      4186
         weighted avg     0.7813    0.7785    0.7728      4186

Confusion Matrix:
[[1080  655]
 [ 272 2179]]

========== REDDIT TEST RESULTS ==========
                 

In [30]:
# Inspect the original student dataset columns
print("Original student columns:")
print(student_df.columns.tolist())

print("\nStudent text examples:")
print(student_df['text'].head(5).to_list())

print("\nCheck whether target column appears in text:")
print(
    student_df.apply(
        lambda row: str(row['Depression']) in str(row['text']),
        axis=1
    ).value_counts()
)

Original student columns:
['Gender', 'Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Sleep Duration', 'Dietary Habits', 'Work/Study Hours', 'Financial Stress', 'Family History of Mental Illness', 'Depression', 'text']

Student text examples:
["[CLINICAL] Subject is a 33-year-old Male. Academic standing: CGPA of 4.49. They report extreme academic pressure and no work pressure. Sleep duration: '5-6 hours'. Dietary habits: Healthy. Financial stress is very low. Family history of mental illness: No.", "[CLINICAL] Subject is a 24-year-old Female. Academic standing: CGPA of 2.95. They report low academic pressure and no work pressure. Sleep duration: '5-6 hours'. Dietary habits: Moderate. Financial stress is low. Family history of mental illness: Yes.", "[CLINICAL] Subject is a 31-year-old Male. Academic standing: CGPA of 3.52. They report moderate academic pressure and no work pressure. Sleep duration: 'Less than 5 hours'. Dietary habits: Healthy. Financial stress is very low. Family 

In [31]:
# Inspect the original student dataset columns
print("Original student columns:")
print(student_df.columns.tolist())

print("\nStudent text examples:")
print(student_df['text'].head(5).to_list())

print("\nCheck whether target column appears in text:")
print(
    student_df.apply(
        lambda row: str(row['Depression']) in str(row['text']),
        axis=1
    ).value_counts()
)

Original student columns:
['Gender', 'Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Sleep Duration', 'Dietary Habits', 'Work/Study Hours', 'Financial Stress', 'Family History of Mental Illness', 'Depression', 'text']

Student text examples:
["[CLINICAL] Subject is a 33-year-old Male. Academic standing: CGPA of 4.49. They report extreme academic pressure and no work pressure. Sleep duration: '5-6 hours'. Dietary habits: Healthy. Financial stress is very low. Family history of mental illness: No.", "[CLINICAL] Subject is a 24-year-old Female. Academic standing: CGPA of 2.95. They report low academic pressure and no work pressure. Sleep duration: '5-6 hours'. Dietary habits: Moderate. Financial stress is low. Family history of mental illness: Yes.", "[CLINICAL] Subject is a 31-year-old Male. Academic standing: CGPA of 3.52. They report moderate academic pressure and no work pressure. Sleep duration: 'Less than 5 hours'. Dietary habits: Healthy. Financial stress is very low. Family 

In [32]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

final_model_path = (
    "/content/gdrive/MyDrive/mental_health_prediction/final_unified_model"
)

# Confirm the folder exists
assert os.path.exists(final_model_path), "Model folder not found!"

# Load the saved tokenizer
loaded_tokenizer = AutoTokenizer.from_pretrained(final_model_path)

# Load the saved classification model
loaded_model = AutoModelForSequenceClassification.from_pretrained(
    final_model_path
)

loaded_model.eval()

print("Model loaded successfully!")
print("Model architecture:", loaded_model.__class__.__name__)
print("Number of labels:", loaded_model.config.num_labels)
print("Labels:", loaded_model.config.id2label)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully!
Model architecture: RobertaForSequenceClassification
Number of labels: 2
Labels: {0: 'No identified concern', 1: 'Mental-health concern'}
